# Explainable Agentic RAG Walkthrough

## Goal

This companion notebook explains *why* each knowledge-base paragraph is retrieved.
The production implementation remains in the `agentic_rag` package and CLI;
this notebook imports that code rather than duplicating it.

## Setup

Run from the repository root after `pip install -e ".[notebook]"`.
Retrieval cells require no API key. The optional final cell runs the two-agent
workflow only when `OPENAI_API_KEY` is already present in the environment.

In [1]:
import os
import sys
from pathlib import Path

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "knowledge_base.txt").is_file()
)
sys.path.insert(0, str(repo_root))

from agentic_rag.retrieval import BM25Retriever  # noqa: E402

knowledge_base_path = repo_root / "knowledge_base.txt"
retriever = BM25Retriever(
    knowledge_base_path,
    max_results=5,
    min_score=0.15,
    min_query_coverage=0.40,
)
chunks = retriever.load_chunks()
print(f"Loaded {len(chunks)} stable paragraph chunks from {knowledge_base_path.name}")

Loaded 10 stable paragraph chunks from knowledge_base.txt


## Steps

### 1. Retrieve evidence and inspect ranking factors

The ranking is deterministic. Each result exposes its matched query terms,
per-term BM25 contribution, query coverage, and consecutive-phrase bonus.
A 40% coverage gate and mutually exclusive domestic/international qualifier
check suppress chunks that match only generic policy vocabulary. The rarest
query term appearing in section titles becomes an auditable intent anchor.

In [2]:
query = "What approvals and preparations are required for international business travel?"
bundle = retriever.search(query)
print(f"Intent anchor terms: {bundle.intent_anchor_terms}\n")

for chunk in bundle.chunks:
    print(f"{chunk.chunk_id} | {chunk.title} | score={chunk.score:.4f}")
    print(f"  matched_terms={chunk.matched_terms}")
    print(f"  term_contributions={chunk.contribution_dict()}")
    print(f"  coverage={chunk.coverage:.2%}; phrase_bonus={chunk.phrase_bonus:.2f}")
    print(f"  evidence={chunk.text}\n")

Intent anchor terms: ('international',)

KB-003 | International Business Travel | score=4.7316
  matched_terms=('approval', 'international', 'business', 'travel')
  term_contributions={'approval': 0.7634512607740647, 'international': 1.9175278389520012, 'business': 0.9716090559544083, 'travel': 0.8269305021037517}
  coverage=66.67%; phrase_bonus=0.70
  evidence=International business travel requires approval from both the employee's line manager and department head. The request must be submitted at least 14 calendar days before departure and must include the business purpose, destination, dates, and estimated cost.

KB-004 | International Business Travel | score=3.7883
  matched_terms=('international', 'business', 'travel')
  term_contributions={'international': 1.9623984889462556, 'business': 0.5902480966557607, 'travel': 1.0806608147938912}
  coverage=50.00%; phrase_bonus=0.70
  evidence=Before departure, international travelers must complete the travel-risk briefing, confirm emergen

### 2. Verify the no-hallucination retrieval gate

An unrelated query should return no evidence. The workflow then stops before
the Report Generator and returns a deterministic insufficient-information response.

In [3]:
unknown_bundle = retriever.search("What is the company dress code?")
print(unknown_bundle.to_agent_text())
assert unknown_bundle.chunks == ()

NO_RELEVANT_EVIDENCE


## Checks

These checks make the core retrieval expectations executable rather than
relying on screenshots or prose.

In [4]:
assert bundle.has_evidence
assert bundle.chunks[0].title == "International Business Travel"
assert all(chunk.score >= retriever.min_score for chunk in bundle.chunks)
assert len(bundle.source_ids) == len(set(bundle.source_ids))
print("Explainability checks passed.")

Explainability checks passed.


### Optional: run the complete two-agent workflow

This cell is credential-aware so the notebook remains reproducible in CI.
It never prompts for or displays a secret.

In [5]:
if os.getenv("OPENAI_API_KEY"):
    from agentic_rag.agents import build_agents
    from agentic_rag.config import Settings
    from agentic_rag.workflow import AgenticRAGWorkflow

    settings = Settings.from_env()
    workflow = AgenticRAGWorkflow(retriever=retriever, agents=build_agents(settings))
    live_result = await workflow.run(query)
    print(live_result.report.answer_markdown)
else:
    print("Skipped live agent call: OPENAI_API_KEY is not set.")

Skipped live agent call: OPENAI_API_KEY is not set.


## Next Steps

For the submission demo, run `python main.py --demo` with a locally exported
API key. The CLI displays the same evidence IDs and scores before the grounded
final answer, making the end-to-end decision path auditable.